[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P11/P11-1_Structure_CORRIGE.ipynb)

# Algèbre linéaire et structures

Chapitre 11 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre bureau d'études vérifie un support, un treillis, une ossature : chaque barre porte un effort, chaque nœud se déplace, et tout dépend de tout ? Vous voudriez que l'ordinateur résolve l'ensemble d'un coup, qu'il dise si sa réponse est fiable, et qu'il rende aussi les fréquences auxquelles la structure se met à vibrer ?

**Objectif terminal.** Ce chapitre montre comment, avec les matrices de NumPy et de SciPy.

**Objectif instrumental.**

- Résoudre un système linéaire et juger sa fiabilité par le conditionnement.
- Assembler la matrice de rigidité d'un treillis et lire déplacements et efforts.
- Calculer valeurs propres, modes et fréquences propres avec `eigh`.
- Résoudre un grand système avec une matrice creuse.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 3 à 6 et 10, après une filière scientifique.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 11.1 | Quand la barre se couche | 15 min |
| 11.2 | Efforts du treillis de Warren | 25 min |
| 11.3 | Fréquences propres de deux masses | 15 min |
| 11.4 | Une poutre à 2 001 nœuds (défi) | 20 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [1]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("scipy", version("scipy"))

Python 3.12.0
numpy 2.4.6
scipy 1.18.0


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [2]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [3]:
import math
import numpy as np
import scipy.linalg
from scipy import sparse
from scipy.sparse.linalg import spsolve

### Section 11.1 : un système linéaire se résout d'un coup.

In [4]:
# notion: linalg
# Un nœud porte une charge verticale de 10 kN. Deux barres le tiennent : l'une inclinée de 30 degrés, l'autre horizontale.
# Équilibre du nœud : N1 sin(t) = F (vertical) et N1 cos(t) + N2 = 0 (horizontal), N positif en traction.
F_N = 10000.0
theta = math.radians(30.0)
A = np.array([[math.sin(theta), 0.0],
              [math.cos(theta), 1.0]])
b = np.array([F_N, 0.0])
efforts = np.linalg.solve(A, b)                       # résout A x = b d'un coup, sans calculer l'inverse
efforts_scipy = scipy.linalg.solve(A, b)              # même résultat par SciPy
print("efforts dans les barres (kN) :", np.round(efforts / 1e3, 3))
assert np.allclose(efforts, efforts_scipy), "NumPy et SciPy donnent les mêmes efforts"
assert (
    np.allclose(efforts, [F_N / math.sin(theta), -F_N / math.tan(theta)])
), "formules de l'équilibre"
assert abs(efforts[0] - 20000.0) < 1e-6, "barre inclinée à 30 degrés : 20 kN en traction"

efforts dans les barres (kN) : [ 20.    -17.321]


In [5]:
assert np.allclose(A @ efforts, b), "le résidu A x - b est nul"
# notion: conditionnement
residu = np.linalg.norm(A @ efforts - b)
print("résidu sous 1e-9 N :", residu < 1e-9, "; conditionnement :", round(float(np.linalg.cond(A)), 2))
assert residu < 1e-9 and np.linalg.cond(A) > 1.0, "un conditionnement vaut au moins 1"

résidu sous 1e-9 N : True ; conditionnement : 3.73


## Exercice 11.1 : Quand la barre se couche (15 min)

Reprenez le support à deux barres et calculez le conditionnement pour une barre inclinée de $60$, $30$, $5$ et $1$ degré. Pour chaque angle, ajoutez $0{,}001$ au coefficient $\sin\theta$ de la matrice (une cote d'implantation légèrement fausse) et résolvez à nouveau avec la même charge. L'amplification est l'erreur relative sur les efforts divisée par l'erreur relative sur la matrice, en norme euclidienne pour les efforts, spectrale pour la matrice (`np.linalg.norm(M, 2)`). Vérifiez qu'elle reste au plus égale au conditionnement.

**Résultat attendu.** Conditionnements de 1,73, 3,73, 22,90 et 114,59 ; amplifications de 1,41, 2,73, 16,03 et 76,64, jamais au-dessus du conditionnement.

**Seconde voie.** L'effort de la barre inclinée vaut $F/\sin\theta$, soit 20,0 kN à $30$ degrés, et il croît sans limite quand la barre se couche : le mauvais conditionnement dit ce que dit la mécanique.

**Indication.** Une boucle `for` sur les angles construit la matrice $A$, appelle `np.linalg.cond`, puis résout avec $A$ et avec la matrice perturbée $A_p$ (`A_p = A.copy()`, puis `A_p[0, 0] += 0.001`). L'erreur relative des efforts est divisée par `np.linalg.norm(A_p - A, 2) / np.linalg.norm(A, 2)`. Ce rapport croît quand la barre se couche, sans dépasser le conditionnement. Perturber la charge ne montrerait rien : $x = A^{-1} b$ est proportionnel à $b$, et une charge multipliée par $1{,}001$ multiplie les efforts par $1{,}001$, soit une amplification de $1$ à tout angle. Le risque est dans la matrice.

In [6]:
angles = [60.0, 30.0, 5.0, 1.0]
conditions = []
amplifications = []
# Solution :
for angle in angles:
    t = math.radians(angle)
    At = np.array([[math.sin(t), 0.0], [math.cos(t), 1.0]])
    conditions.append(float(np.linalg.cond(At)))
    x = np.linalg.solve(At, b)
    At_p = At.copy()
    At_p[0, 0] += 0.001                                # cote d'implantation légèrement fausse
    x_perturbe = np.linalg.solve(At_p, b)
    erreur_x = np.linalg.norm(x_perturbe - x) / np.linalg.norm(x)
    erreur_A = np.linalg.norm(At_p - At, 2) / np.linalg.norm(At, 2)
    amplifications.append(float(erreur_x / erreur_A))
    # contre-épreuve : perturber la charge seule donne toujours une amplification de 1
    x_charge = np.linalg.solve(At, b * 1.001)
    amplification_charge = np.linalg.norm(x_charge - x) / np.linalg.norm(x) / 0.001
    assert abs(amplification_charge - 1.0) < 1e-9, "charge : amplification de 1"
    print(f"angle {angle:4.0f} degrés : cond {conditions[-1]:7.2f} ; effort N1 {x[0] / 1e3:8.2f} kN ; "
          f"amplification {amplifications[-1]:6.2f}")
    assert amplifications[-1] <= conditions[-1] + 1e-9, "l'erreur est amplifiée d'au plus cond"
# Fin de la solution
assert conditions == sorted(conditions), "plus la barre se couche, plus le conditionnement croît"
assert amplifications == sorted(amplifications), "l'amplification croît aussi avec l'angle"
assert (
    abs(conditions[0] - 1.73) < 0.005 and abs(conditions[2] - 22.90) < 0.005
), "cond à 60 et 5 degrés"
assert abs(conditions[3] - 114.59) < 0.005, "cond à 1 degré attendu : 114,59"
print("Exercice 11.1 : contrôles passés")

angle   60 degrés : cond    1.73 ; effort N1    11.55 kN ; amplification   1.41
angle   30 degrés : cond    3.73 ; effort N1    20.00 kN ; amplification   2.73
angle    5 degrés : cond   22.90 ; effort N1   114.74 kN ; amplification  16.03
angle    1 degrés : cond  114.59 ; effort N1   572.99 kN ; amplification  76.64
Exercice 11.1 : contrôles passés


### Section 11.2 : une structure s'assemble en matrice de rigidité.

In [7]:
# notion: rigidite
# Treillis de Warren : cinq nœuds, sept barres articulées, acier E = 210 000 MPa, section de 300 mm2.
# Appui fixe au nœud 1, appui mobile au nœud 3, charge de 20 kN vers le bas au nœud 2.
F_treillis_N, pas_mm, haut_mm, E_MPa, section_mm2 = 20000.0, 2000.0, 1500.0, 210000.0, 300.0
noeuds = np.array([[0.0, 0.0], [pas_mm, 0.0], [2 * pas_mm, 0.0], [pas_mm / 2, haut_mm], [3 * pas_mm / 2, haut_mm]])
barres = [(0, 1), (1, 2), (3, 4), (0, 3), (3, 1), (1, 4), (4, 2)]       # indices Python, nœuds numérotés de 0 à 4

def geometrie(i, j):
    """Longueur et cosinus directeurs (c, s) de la barre qui va du nœud i au nœud j."""
    d = noeuds[j] - noeuds[i]
    L = math.hypot(d[0], d[1])
    return L, d[0] / L, d[1] / L

def rigidite_barre(i, j):
    """Matrice 4 x 4 d'une barre articulée dans le repère général : E A / L fois les cosinus directeurs."""
    L, c, s = geometrie(i, j)
    k = E_MPa * section_mm2 / L * np.array([[c * c, c * s], [c * s, s * s]])
    return np.block([[k, -k], [-k, k]])

In [8]:
K = np.zeros((10, 10))                              # deux degrés de liberté par nœud : 10 inconnues

for i, j in barres:
    ddl = [2 * i, 2 * i + 1, 2 * j, 2 * j + 1]      # degrés de liberté des deux nœuds de la barre
    K[np.ix_(ddl, ddl)] += rigidite_barre(i, j)     # chaque barre ajoute son bloc aux lignes et colonnes de ses nœuds

print("matrice assemblée :", K.shape, "; termes non nuls :", np.count_nonzero(K))
assert K.shape == (10, 10) and np.allclose(K, K.T), "la matrice est symétrique"
assert np.count_nonzero(K) == 52, "52 termes non nuls"
appuis = [0, 1, 5]                                  # degrés de liberté bloqués : 1x, 1y, 3y
libres = [d for d in range(10) if d not in appuis]
K_reduite = K[np.ix_(libres, libres)]
f = np.zeros(10)
f[3] = -F_treillis_N                                # le nœud 2 (indice 1) porte la charge selon y, vers le bas
print("matrice réduite :", K_reduite.shape)
assert K_reduite.shape == (7, 7)
u = np.zeros(10)
u[libres] = np.linalg.solve(K_reduite, f[libres])

matrice assemblée : (10, 10) ; termes non nuls : 52
matrice réduite : (7, 7)


In [9]:
print(f"flèche du nœud 2 : {u[3]:.4f} mm")
reactions = K @ u - f                               # forces de liaison : K u moins les charges
print("réactions (kN) :", np.round(reactions[appuis] / 1e3, 3) + 0.0)
assert (
    abs(reactions[[1, 5]].sum() - F_treillis_N) < 1e-6
), "les réactions verticales équilibrent la charge"
assert abs(reactions[0]) < 1e-6, "aucune charge horizontale"

flèche du nœud 2 : -1.2500 mm
réactions (kN) : [ 0. 10. 10.]


## Exercice 11.2 : Efforts du treillis de Warren (25 min)

À partir des déplacements $u$ du texte, calculez l'effort normal de chacune des sept barres : allongement $\Delta = c\,(u_{j,x} - u_{i,x}) + s\,(u_{j,y} - u_{i,y})$, effort $E\,A\,\Delta/L$. Calculez aussi le conditionnement de la matrice réduite. Quelle barre porte le plus ? Comparez avec l'équilibre des nœuds du chapitre 10.

**Résultat attendu.** Flèche de $-1{,}25$ mm au nœud 2 ; efforts, dans l'ordre des barres, de 6,67, 6,67, $-13{,}33$, $-12{,}02$, 12,02, 12,02 et $-12{,}02$ kN, en traction positive ; conditionnement de 20.

**Seconde voie.** Le treillis est isostatique : l'équilibre des nœuds (section 10.3) donne les mêmes sept efforts sans aucune matrice de rigidité ; les déplacements, eux, demandent la rigidité.

**Indication.** Une boucle `for` sur `barres` appelle `geometrie`, calcule l'allongement avec les déplacements de `u`, puis l'effort $E\,A\,\Delta/L$, converti en kilonewtons. La barre 3, la membrure supérieure, porte le plus, en compression. `np.linalg.cond(K_reduite)` donne le conditionnement.

Les contrôles de la cellule lisent la variable `conditionnement_treillis` : gardez ce nom.

In [10]:
efforts_barres = []
# Solution :
for i, j in barres:
    L, c, s = geometrie(i, j)
    allongement = c * (u[2 * j] - u[2 * i]) + s * (u[2 * j + 1] - u[2 * i + 1])
    efforts_barres.append(E_MPa * section_mm2 / L * allongement / 1e3)         # N converti en kN

conditionnement_treillis = float(np.linalg.cond(K_reduite))
print("efforts (kN, + traction) :", np.round(efforts_barres, 2))
print(f"conditionnement de la matrice réduite : {conditionnement_treillis:.2f}")
# Fin de la solution
assert abs(u[3] + 1.25) < 0.005, "flèche attendue : -1,25 mm"
assert (
    abs(sum(abs(e) for e in efforts_barres) - 74.7407) < 0.001
), "somme des efforts en valeur absolue"
assert abs(conditionnement_treillis - 20.0) < 0.1, "conditionnement attendu : 20"
print("Exercice 11.2 : contrôles passés")

efforts (kN, + traction) : [  6.67   6.67 -13.33 -12.02  12.02  12.02 -12.02]
conditionnement de la matrice réduite : 19.95
Exercice 11.2 : contrôles passés


### Section 11.3 : les modes propres sont des valeurs propres.

In [11]:
# notion: valeurs-propres
S = np.array([[3.0, 1.0], [1.0, 3.0]])              # matrice symétrique
valeurs, vecteurs = np.linalg.eigh(S)               # valeurs propres rangées en croissant, vecteurs en colonnes
print("valeurs propres :", valeurs)
assert np.allclose(valeurs, [2.0, 4.0]), "valeurs propres de S : 2 et 4"

for k in range(2):
    assert (
        np.allclose(S @ vecteurs[:, k], valeurs[k] * vecteurs[:, k])
    ), "S v = lambda v : direction qui ne tourne pas"

assert np.allclose(vecteurs.T @ vecteurs, np.eye(2)), "les axes propres sont orthogonaux"

valeurs propres : [2. 4.]


In [12]:
# notion: modes
# Chaîne mur, ressort k1, masse m1, ressort k2, masse m2 : M u'' + K u = 0, soit K v = lambda M v avec lambda = (2 pi f)^2.
masses = np.diag([5.0, 3.0])                                      # kg
raideurs = np.array([[3.0e6, -1.0e6], [-1.0e6, 1.0e6]])           # N/m : k1 + k2, -k2, -k2, k2 avec k1 = 2e6, k2 = 1e6
lambdas, formes = scipy.linalg.eigh(raideurs, masses)             # problème généralisé, formes normalisées par la masse
frequences_Hz = np.sqrt(lambdas) / (2 * math.pi)
print("fréquences propres (Hz) :", np.round(frequences_Hz, 2))
assert (
    np.allclose(formes.T @ masses @ formes, np.eye(2))
), "formes propres orthonormées pour la matrice de masse"
assert np.all(np.diff(frequences_Hz) > 0)

fréquences propres (Hz) : [ 66.78 138.5 ]


In [13]:
# notion: covariance
# Étapes de l'auteur (analyse en composantes principales) : centrer, standardiser à la main, diagonaliser, tourner le
# repère avec les vecteurs propres, cumuler les valeurs propres. Noms de l'auteur : eigenvalues, eigenvectors.
rng = np.random.default_rng(2026)                    # graine fixée : mêmes tirages à chaque exécution
n_points = 200
x1 = rng.normal(0.0, 1.0, n_points)
x2 = 0.8 * x1 + rng.normal(0.0, 0.5, n_points)       # deux grandeurs corrélées
x3 = rng.uniform(-1.0, 1.0, n_points)                # une grandeur indépendante
X = np.column_stack([x1, x2, x3])
centre = X - X.mean(axis=0)                          # centrer : retirer la moyenne de chaque colonne
cov_main = centre.T @ centre / (n_points - 1)        # covariance écrite à la main
cov = np.cov(X, rowvar=False)                        # np.cov : une variable par colonne si rowvar=False
standardise = centre / X.std(axis=0, ddof=1)         # standardiser à la main : moyenne nulle, écart-type 1
correlation = np.cov(standardise, rowvar=False)      # la covariance des données standardisées est la corrélation
eigenvalues, eigenvectors = np.linalg.eigh(cov)
eigenvalues, eigenvectors = eigenvalues[::-1], eigenvectors[:, ::-1]     # du plus grand au plus petit

In [14]:
scores = np.dot(centre, eigenvectors)                # rotation du repère sur les axes propres
cov_scores = np.cov(scores, rowvar=False)
variance_cumulee = np.cumsum(eigenvalues) / eigenvalues.sum()
print("valeurs propres :", np.round(eigenvalues, 3), "; part cumulée :", np.round(variance_cumulee, 3))
assert np.allclose(cov, cov_main), "np.cov redonne la covariance écrite à la main"
assert (
    np.allclose(np.diag(correlation), 1.0) and abs(correlation[0, 1] - 0.875) < 0.0005
), "corrélation des deux premières"
assert (
    np.allclose(cov_scores, np.diag(eigenvalues), atol=1e-12)
), "dans le repère propre, les covariances s'annulent"
assert (
    np.isclose(np.trace(cov), eigenvalues.sum())
), "la variance totale ne change pas par rotation"

valeurs propres : [2.109 0.326 0.139] ; part cumulée : [0.819 0.946 1.   ]


In [15]:
assert (
    abs(variance_cumulee[0] - 0.819) < 0.0005 and np.isclose(variance_cumulee[-1], 1.0)
), "part cumulée"
part_cumulee = variance_cumulee

## Exercice 11.3 : Fréquences propres de deux masses (15 min)

Le système du chapitre 10 compte deux masses, $m_1 = 5$ kg et $m_2 = 3$ kg, et deux ressorts, $k_1 = 2\times10^6$ N/m entre le mur et la masse 1, $k_2 = 10^6$ N/m entre les deux masses. Écrivez $M$ et $K$, calculez les deux fréquences propres avec `scipy.linalg.eigh`, et dites si, au premier mode, les masses vont dans le même sens.

**Résultat attendu.** Fréquences de 66,78 Hz et 138,5 Hz ; au premier mode, les deux masses vont dans le même sens.

**Seconde voie.** L'équation caractéristique en lettres du chapitre 10 donne 66,78 et 138,5 Hz : le calcul numérique retrouve le calcul littéral.

**Indication.** La matrice des masses est `np.diag([5.0, 3.0])`. La matrice des raideurs est `[[k1 + k2, -k2], [-k2, k2]]`. `lambdas, formes = scipy.linalg.eigh(K, M)` rend $\lambda = \omega^2$ et les modes ; la fréquence est `np.sqrt(lambdas) / (2 * math.pi)`. Les composantes de la première colonne de `formes` ont le même signe.

Les contrôles de la cellule lisent les variables `f1_Hz`, `f2_Hz`, `mode_1` : gardez ces noms.

In [16]:
# Solution :
f1_Hz, f2_Hz = float(frequences_Hz[0]), float(frequences_Hz[1])
mode_1 = formes[:, 0] / formes[:, 0][np.argmax(np.abs(formes[:, 0]))]     # normalisé : la plus grande composante vaut 1
print(f"fréquences : {f1_Hz:.2f} Hz et {f2_Hz:.1f} Hz ; premier mode : {np.round(mode_1, 3)}")
# Fin de la solution
assert (
    abs(f1_Hz - 66.78) < 0.005 and abs(f2_Hz - 138.5) < 0.05
), "fréquences attendues : 66,78 et 138,5 Hz"
assert mode_1[0] * mode_1[1] > 0, "au premier mode, les deux masses vont dans le même sens"
assert np.isclose(np.abs(mode_1).max(), 1.0)
print("Exercice 11.3 : contrôles passés")

fréquences : 66.78 Hz et 138.5 Hz ; premier mode : [0.472 1.   ]
Exercice 11.3 : contrôles passés


### Section 11.4 : les matrices creuses portent les grands systèmes.

In [17]:
# notion: differences-finies
# Poutre sur deux appuis, charge F au milieu : E I y'' = M(x). Au nœud i, y'' vaut (y[i-1] - 2 y[i] + y[i+1]) / h^2.
F_poutre_N, L_mm, E_poutre_MPa, I_mm4 = 2000.0, 1000.0, 210000.0, 20.0 * 40.0**3 / 12

def fleche_differences_finies(n_noeuds):
    """Flèche au milieu (en mm, positive vers le bas) de la poutre, par différences finies sur n_noeuds nœuds."""
    h = L_mm / (n_noeuds - 1)
    x = np.linspace(0.0, L_mm, n_noeuds)
    moment = F_poutre_N * np.minimum(x, L_mm - x) / 2            # moment fléchissant en N.mm
    n_int = n_noeuds - 2                                         # nœuds intérieurs : y vaut 0 aux deux appuis
    A_dense = (np.diag(-2.0 * np.ones(n_int)) + np.diag(np.ones(n_int - 1), 1) + np.diag(np.ones(n_int - 1), -1)) / h**2
    y = np.linalg.solve(A_dense, moment[1:-1] / (E_poutre_MPa * I_mm4))
    return -float(y[n_int // 2])

erreurs_fd = {n: abs(fleche_differences_finies(n) - 1.8601190476) for n in (11, 21, 41)}
print("erreur de la flèche (mm) pour 11, 21 et 41 nœuds :", {n: round(e, 6) for n, e in erreurs_fd.items()})
assert erreurs_fd[11] > erreurs_fd[21] > erreurs_fd[41], "l'erreur décroît avec le pas"

erreur de la flèche (mm) pour 11, 21 et 41 nœuds : {11: 0.037202, 21: 0.009301, 41: 0.002325}


In [18]:
assert (
    abs(erreurs_fd[21] / erreurs_fd[41] - 4.0) < 0.5
), "l'erreur est divisée par quatre quand le pas est divisé par deux"
# notion: sparse
n_noeuds = 2001
n_int = n_noeuds - 2
pas_h = L_mm / (n_noeuds - 1)
x_noeuds = np.linspace(0.0, L_mm, n_noeuds)
moment_Nmm = F_poutre_N * np.minimum(x_noeuds, L_mm - x_noeuds) / 2
A_creuse = sparse.diags([np.ones(n_int - 1), -2.0 * np.ones(n_int), np.ones(n_int - 1)], [-1, 0, 1], format="csc") / pas_h**2
y_creuse = spsolve(A_creuse, moment_Nmm[1:-1] / (E_poutre_MPa * I_mm4))
octets_dense = n_int * n_int * 8
octets_creuse = A_creuse.data.nbytes + A_creuse.indices.nbytes + A_creuse.indptr.nbytes
print(f"matrice {n_int} x {n_int} : {A_creuse.nnz} termes non nuls ; plein : {octets_dense / 1e6:.1f} Mo ; creux : {octets_creuse / 1e3:.0f} ko")
assert A_creuse.shape == (1999, 1999) and A_creuse.nnz == 3 * n_int - 2, "trois diagonales"
assert octets_creuse * 100 < octets_dense, "la matrice creuse occupe cent fois moins de mémoire"

matrice 1999 x 1999 : 5995 termes non nuls ; plein : 32.0 Mo ; creux : 80 ko


## Exercice 11.4 : Une poutre à 2 001 nœuds (défi, 20 min)

Reprenez la poutre avec $2\,001$ nœuds, construisez la matrice creuse et résolvez avec `spsolve`. Retrouvez la flèche au milieu, comparez-la à la formule (chapitre 3), puis comptez termes non nuls et mémoire d'une matrice pleine. Vérifiez qu'avec $401$ nœuds en matrice pleine la flèche est à $10^{-4}$ mm près.

**Résultat attendu.** Flèche de 1,8601 mm avec 2 001 nœuds, contre 1,8601 mm par la formule ; 5 995 termes non nuls, contre 32 Mo en matrice pleine ; avec $401$ nœuds en matrice pleine, écart sous $10^{-4}$ mm.

**Seconde voie.** La formule $F L^3/(48\,E I)$ du chapitre 3 donne 1,8601 mm ; l'écart d'un facteur quatre à chaque doublement du nombre de nœuds, de 0,0372 à 0,0023 mm, annonce que le raffinement converge vers cette valeur.

**Indication.** Reprenez `sparse.diags` avec $n = 1\,999$ nœuds intérieurs et le moment du texte, puis `spsolve`. La flèche est $-y$ au nœud du milieu, d'indice $n // 2$. Le nombre de termes non nuls vaut $3n - 2$, la mémoire pleine $8\,n^2$ octets. Avec `fleche_dense(401)`, l'écart reste sous $10^{-4}$ mm.

Les contrôles de la cellule lisent les variables `fleche_creuse_mm`, `fleche_exacte_mm` : gardez ces noms.

In [19]:
# Solution :
fleche_creuse_mm = -float(y_creuse[(n_noeuds - 2) // 2])
fleche_exacte_mm = F_poutre_N * L_mm**3 / (48 * E_poutre_MPa * I_mm4)
print(f"flèche : {fleche_creuse_mm:.4f} mm ; exacte : {fleche_exacte_mm:.4f} mm ; écart sous 1e-5 mm :", abs(fleche_creuse_mm - fleche_exacte_mm) < 1e-5)
# Fin de la solution
assert abs(fleche_creuse_mm - 1.8601) < 5e-5, "flèche attendue : 1,8601 mm"
assert abs(fleche_creuse_mm - fleche_exacte_mm) < 1e-5, "accord avec la formule exacte"
assert (
    abs(fleche_differences_finies(401) - fleche_creuse_mm) < 1e-4
), "matrice pleine et matrice creuse concordent"
print("Exercice 11.4 : contrôles passés")

flèche : 1.8601 mm ; exacte : 1.8601 mm ; écart sous 1e-5 mm : True
Exercice 11.4 : contrôles passés


## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F11-1 : Systèmes linéaires (`numpy.linalg.solve`)

In [20]:
import numpy as np

K = np.array([[30e3, -10e3],            # raideurs sur le chariot 1, N/m
              [-10e3, 40e3]])           # raideurs sur le chariot 2, N/m
F = np.array([400.0, 800.0])            # efforts exercés, N
u = np.linalg.solve(K, F)               # déplacements des chariots, m
print(f"u1 = {u[0]*1e3:.3f} mm, u2 = {u[1]*1e3:.3f} mm".replace(".", ","))
print("Vérification :", np.allclose(K @ u, F))

u1 = 21,818 mm, u2 = 25,455 mm
Vérification : True


### Fiche F11-2 : Valeurs propres (`scipy.linalg.eigh`)

In [21]:
import numpy as np
import scipy.linalg

m, k = 50e3, 20e6                         # masse 50 t, raideur 20 MN/m
K = np.array([[2*k, -k, 0], [-k, 2*k, -k], [0, -k, k]])
M = np.diag([m, m, m])                    # masses par étage, kg
lambdas, V = scipy.linalg.eigh(K, b=M)
f = np.sqrt(lambdas) / (2 * np.pi)        # fréquences propres, Hz
txt = " ".join(f"{x:.2f}" for x in f)
print("fréquences (Hz) :", txt.replace(".", ","))
print("orthogonales :", np.allclose(V.T @ M @ V, np.eye(3)))

fréquences (Hz) : 1,42 3,97 5,74
orthogonales : True
